In [ ]:
## ============================================================
# DEEP_DIVE_4CONN_V2: 4 ТОЧКИ (500 тыс ШАГОВ) С ЧЕКПОИНТАМИ
# ============================================================

import numpy as np
from scipy.ndimage import label
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import csv
import time
import os
from IPython.display import display, Image as IPImage

# ------------------------------------------------------------
# REM: ПАПКИ
# ------------------------------------------------------------
output_dir = "DEEP_DIVE_4CONN_V2"
csv_dir = os.path.join(output_dir, "csv")
png_dir = os.path.join(output_dir, "png")
ckpt_dir = os.path.join(output_dir, "checkpoints")
os.makedirs(output_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)
os.makedirs(png_dir, exist_ok=True)
os.makedirs(ckpt_dir, exist_ok=True)

# ------------------------------------------------------------
# REM: ПАРАМЕТРЫ
# ------------------------------------------------------------
L = 100
STEPS = 500_000  # 500 тыс шагов
beta = 1.0
lam = 3.0          

POINTS = [
    (0.10, 2.0),  
    (0.10, 2.5),  
    (0.50, 1.5),  
    (0.50, 2.0)   
]

csv_file = os.path.join(csv_dir, "deep_dive_4conn_v2.csv")

if not os.path.exists(csv_file):
    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['p_init', 'lam', 'sigma', 'step', 'density', 'P_inf', 'perimeter', 'max_size'])

# ------------------------------------------------------------
# REM: ФИЗИЧЕСКИЕ ФУНКЦИИ
# ------------------------------------------------------------
def init_lattice(L, p):
    return (np.random.random((L, L)) < p).astype(int)

def find_clusters_periodic(grid):
    padded = np.pad(grid, pad_width=1, mode='wrap')
    labeled, n = label(padded, structure=[[0,1,0],[1,1,1],[0,1,0]])
    labeled = labeled[1:-1, 1:-1]
    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0
    return labeled, sizes

def calculate_perimeter_periodic(grid):
    right = np.roll(grid, 1, axis=1)
    left = np.roll(grid, -1, axis=1)
    up = np.roll(grid, 1, axis=0)
    down = np.roll(grid, -1, axis=0)
    p = 0
    p += np.sum((grid == 1) & (right == 0))
    p += np.sum((grid == 1) & (left == 0))
    p += np.sum((grid == 1) & (up == 0))
    p += np.sum((grid == 1) & (down == 0))
    return int(p)

def action(grid, lam, sigma):
    _, sizes = find_clusters_periodic(grid)
    S_bulk = -lam * np.sum(sizes**2)
    S_surface = sigma * calculate_perimeter_periodic(grid)
    return S_bulk + S_surface

def metropolis_step_swap(grid, lam, sigma, beta=1.0):
    L = grid.shape[0]
    while True:
        i1, j1 = np.random.randint(0, L, 2)
        if grid[i1, j1] == 1: break
    while True:
        i2, j2 = np.random.randint(0, L, 2)
        if grid[i2, j2] == 0: break

    S_old = action(grid, lam, sigma)
    grid[i1, j1] = 0
    grid[i2, j2] = 1
    S_new = action(grid, lam, sigma)
    delta_S = S_new - S_old

    if delta_S <= 0 or np.random.random() < np.exp(-beta * delta_S):
        return True
    else:
        grid[i1, j1] = 1
        grid[i2, j2] = 0
        return False

def measure(grid):
    labeled, sizes = find_clusters_periodic(grid)
    sizes_nonzero = sizes[sizes > 0]
    if len(sizes_nonzero) == 0: return None
    P_inf = float(sizes_nonzero.max() / np.sum(sizes_nonzero))
    density = float(np.mean(grid))
    perimeter = calculate_perimeter_periodic(grid)
    max_size = int(sizes_nonzero.max())
    return {'density': density, 'P_inf': P_inf, 'perimeter': perimeter, 'max_size': max_size}

# ------------------------------------------------------------
# REM: ЗАПУСК С ПОДДЕРЖКОЙ ВОССТАНОВЛЕНИЯ
# ------------------------------------------------------------
for p, sigma in POINTS:
    ckpt_grid_file = os.path.join(ckpt_dir, f"grid_p{p}_s{sigma}.npy")
    ckpt_step_file = os.path.join(ckpt_dir, f"step_p{p}_s{sigma}.npy")
    
    start_step = 0
    grid = None
    
    # REM: Проверяем наличие сохраненной сетки
    if os.path.exists(ckpt_grid_file) and os.path.exists(ckpt_step_file):
        grid = np.load(ckpt_grid_file)
        start_step = int(np.load(ckpt_step_file)[0])
        print(f"\n[RESTART] Плотность {p*100}%, Сигма {sigma}. Восстанавливаем с шага {start_step*5}...", flush=True)
    elif os.path.exists(csv_file):
        # REM: Проверяем, не завершена ли точка
        is_done = False
        with open(csv_file, 'r') as f:
            reader = csv.reader(f)
            next(reader, None)
            for row in reader:
                if len(row) >= 4 and abs(float(row[0]) - p) < 1e-5 and abs(float(row[2]) - sigma) < 1e-5:
                    if int(row[3]) == STEPS * 5: is_done = True
        if is_done:
            print(f"[SKIP] Плотность {p*100}%, Сигма {sigma} уже полностью посчитана.", flush=True)
            continue
        else:
            grid = init_lattice(L, p)
    else:
        grid = init_lattice(L, p)

    print("\n" + "="*65, flush=True)
    print(f"СЧИТАЕМ (4-связность): Плотность = {p*100}%, Сигма = {sigma}, Лямбда = {lam}", flush=True)
    print("="*65, flush=True)
    print(f"{'Шаг':>8} | {'density':>8} | {'P_inf':>8} | {'Perimeter':>10} | {'Max_Size':>8} | {'Time(s)':>7}", flush=True)
    print("-" * 65, flush=True)

    t0 = time.time()
    
    # REM: Цикл начинается с start_step, если произошло восстановление
    for step in range(start_step, STEPS):
        for _ in range(5):
            metropolis_step_swap(grid, lam, sigma, beta)
        
        real_step = step * 5
        
        # REM: Сохраняем чекпоинт каждые 250 тыс. шагов
        if real_step % 250_000 == 0 and real_step > 0:
            m = measure(grid)
            elapsed = time.time() - t0
            
            with open(csv_file, 'a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([p, lam, sigma, real_step, m['density'], m['P_inf'], m['perimeter'], m['max_size']])
            
            print(f"{real_step:>8} | {m['density']:>8.3f} | {m['P_inf']:>8.3f} | {m['perimeter']:>10.1f} | {m['max_size']:>8d} | {elapsed:>7.1f}", flush=True)
            
            # Сохраняем сетку и шаг на диск
            np.save(ckpt_grid_file, grid)
            np.save(ckpt_step_file, np.array([step]))
            
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(grid, cmap='binary', interpolation='nearest')
            ax.set_title(f'4-conn D={p*100}%, σ={sigma}, Step={real_step}')
            img_path = os.path.join(png_dir, f"C_D{p}_S{sigma}_step{real_step}.png")
            plt.savefig(img_path, dpi=100, bbox_inches='tight')
            plt.close(fig)
            display(IPImage(filename=img_path))

    # REM: Финальный замер и очистка чекпоинтов
    m = measure(grid)
    with open(csv_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([p, lam, sigma, STEPS*5, m['density'], m['P_inf'], m['perimeter'], m['max_size']])
    
    if os.path.exists(ckpt_grid_file): os.remove(ckpt_grid_file)
    if os.path.exists(ckpt_step_file): os.remove(ckpt_step_file)
    
    print(f"ТОЧКА ЗАВЕРШЕНА.", flush=True)

print("\nВСЕ 4 ТОЧКИ 4-СВЯЗНОСТИ ГОТОВЫ! Данные в DEEP_DIVE_4CONN_V2.", flush=True)